# 05 — Feature Engineering

## Objective
Create useful features for predicting next-week HAB severity
and remove highly redundant predictors.

This notebook will:
1. Review existing features.
2. Engineer HAB history and trend features.
3. Engineer seasonal features.
4. Engineer selected environmental features.
5. Identify and remove highly redundant predictors.
6. Validate and export the feature-engineered dataset.


## 1. Load the Weekly Modeling Dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")


Libraries imported successfully!


In [4]:
data_path = "../data/model_table_weekly.parquet"

# Load the dataset
df = pd.read_parquet(data_path)

# Create a working copy for feature engineering
df_fe = df.copy()

# Display dataset information
print("Dataset loaded successfully!")
print("Dataset location:", data_path)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Preview the first five rows
display(df.head())



Dataset loaded successfully!
Dataset location: ../data/model_table_weekly.parquet
Number of rows: 366
Number of columns: 47


,week_start,region,split,y_class,y_cellcount_max,n_habsos_samples,low_coverage,hab_class,hab_cellcount_max,nutr_chla_corr_ugL,nutr_cond_umho,nutr_do_mgL,nutr_do_sat_pct,nutr_nh3_mgL,nutr_nox_mgL,nutr_ph,nutr_po4_mgL,nutr_tkn_mgL,nutr_tn_mgL,nutr_tp_mgL,nutr_tss_mgL,nutr_turbidity_ntu,nutr_water_temp_c,has_nutrients,rain_total_mm,air_tmax_c,air_tmin_c,air_tmean_c,ksrq_pressure_hpa,ksrq_air_temp_c,ksrq_dew_point_c,ksrq_wind_speed,ksrq_wind_u,ksrq_wind_v,has_ksrq,uwnd_offshore,mote_temp_mean_c,mote_temp_max_c,has_mote,chl_box_median,chl_coastal_median,chl_coastal_anomaly,cyano_samples,cyano_detections,rain_2wk_mm,rain_4wk_mm,rain_8wk_mm
0,2018-12-31,sarasota_box,train,2.0,1303204.0,73.0,0,2.0,186266667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,15.748,28.333333,10.000000,20.793651,NaN,NaN,NaN,NaN,NaN,NaN,0,-0.303916,NaN,NaN,0,0.687255,6.330664,0.061709,8.0,0.0,NaN,NaN,NaN
1,2019-01-07,sarasota_box,train,2.0,2428000.0,107.0,0,2.0,1303204.0,NaN,30314.152083,7.043779,87.531330,NaN,NaN,7.707167,NaN,NaN,NaN,NaN,NaN,NaN,20.004847,1,0.000,26.111111,6.111111,17.142857,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.466936,NaN,NaN,0,0.785405,6.225243,0.550108,8.0,2.0,15.748,NaN,NaN
2,2019-01-14,sarasota_box,train,2.0,282720.0,93.0,0,2.0,2428000.0,NaN,2040.693200,7.269080,76.263102,NaN,NaN,7.520000,NaN,NaN,NaN,NaN,NaN,NaN,18.736990,1,1.524,26.111111,3.333333,13.968254,NaN,NaN,NaN,NaN,NaN,NaN,0,0.662809,NaN,NaN,0,0.746753,5.929194,0.264531,NaN,NaN,1.524,NaN,NaN
3,2019-01-21,sarasota_box,train,0.0,667.0,76.0,0,2.0,282720.0,NaN,6454.335500,6.585240,67.686205,NaN,NaN,7.626818,NaN,NaN,NaN,NaN,NaN,NaN,17.385055,1,56.134,26.111111,4.444444,14.761905,NaN,NaN,NaN,NaN,NaN,NaN,0,-3.750830,NaN,NaN,0,0.869726,6.499056,0.539973,NaN,NaN,57.658,73.406,NaN
4,2019-01-28,sarasota_box,train,0.0,333.0,106.0,0,0.0,667.0,NaN,6454.335500,6.585240,67.686205,NaN,NaN,7.626818,NaN,NaN,NaN,NaN,NaN,NaN,17.385055,0,0.508,26.666667,4.444444,15.476190,NaN,NaN,NaN,NaN,NaN,NaN,0,-2.837600,NaN,NaN,0,0.982401,4.829902,-0.594875,NaN,NaN,56.642,58.166,NaN


#### Display all existing features

In [5]:

# Create a feature inventory
feature_inventory = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

# Start numbering from 1
feature_inventory.index = range(
    1, len(feature_inventory) + 1
)

print("EXISTING FEATURE INVENTORY")
print("-" * 40)

display(feature_inventory)

print("\nTotal columns:", len(df.columns))
print(
    "Duplicate column names:",
    df.columns.duplicated().sum()
)


EXISTING FEATURE INVENTORY
----------------------------------------


,Feature,Data Type
1,week_start,datetime64[ns]
2,region,object
3,split,object
4,y_class,float64
5,y_cellcount_max,float64
6,n_habsos_samples,float64
7,low_coverage,int64
8,hab_class,float64
9,hab_cellcount_max,float64
10,nutr_chla_corr_ugL,float64



Total columns: 47
Duplicate column names: 0


In [6]:

# Keywords commonly used in engineered feature names
engineering_keywords = [
    "lag",
    "rolling",
    "anomaly",
    "2wk",
    "4wk",
    "8wk",
    "sin",
    "cos",
    "change",
    "trend"
]

# Identify candidate engineered features
existing_engineered = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in engineering_keywords
    )
]

print("POTENTIALLY ENGINEERED FEATURES")
print("-" * 40)

for feature in existing_engineered:
    print("-", feature)

print(
    "\nNumber identified:",
    len(existing_engineered)
)


POTENTIALLY ENGINEERED FEATURES
----------------------------------------
- chl_coastal_anomaly
- rain_2wk_mm
- rain_4wk_mm
- rain_8wk_mm

Number identified: 4



## 1. HAB History and Trend Feature Engineering


Now we will be creating new features that capture the current intensity, historical severity, and recent changes in Harmful Algal Bloom (HAB) conditions.

We engineer three candidate features:

1. **hab_cellcount_log:** Log-transformed current-week maximum HAB cell count.
2. **hab_class_lag1:** HAB severity class from the previous week.
3. **hab_log_change_1wk:** Change in log-transformed HAB cell count between the current and previous week.

### Benefits

- Reduce the influence of extremely large cell-count values.
- Capture recent HAB history.
- Identify increasing or decreasing bloom intensity.
- Provide additional temporal information for next-week HAB prediction.

### Missing Values

Missing observations are preserved. No imputation, forward filling, or row removal is performed.


In [7]:
# HAB History and Trend Feature Engineering

# Work with the existing feature-engineering copy
df_fe = df.copy()

# Ensure the date column is datetime
df_fe["week_start"] = pd.to_datetime(
    df_fe["week_start"]
)

# Sort observations chronologically within region
df_fe = df_fe.sort_values(
    ["region", "week_start"]
).reset_index(drop=True)

# Check for duplicate region-week observations
assert not df_fe.duplicated(
    subset=["region", "week_start"]
).any(), "Duplicate region-week observations found."

# Verify consecutive weekly observations
week_gaps = (
    df_fe.groupby("region")["week_start"]
    .diff()
    .dropna()
)

assert week_gaps.eq(
    pd.Timedelta(days=7)
).all(), "Non-consecutive weeks found. Review before creating lags."


# Feature 1: Log HAB cell count

# HAB cell counts should not be negative
assert df_fe["hab_cellcount_max"].dropna().ge(0).all(), (
    "Negative HAB cell counts found."
)

df_fe["hab_cellcount_log"] = np.log1p(
    df_fe["hab_cellcount_max"]
)


# Feature 2: Previous-week HAB class

df_fe["hab_class_lag1"] = (
    df_fe.groupby("region")["hab_class"]
    .shift(1)
)


# Feature 3: One-week HAB log change

# Calculate previous week's log count temporarily
previous_log_count = (
    df_fe.groupby("region")["hab_cellcount_log"]
    .shift(1)
)

# Current week minus previous week
df_fe["hab_log_change_1wk"] = (
    df_fe["hab_cellcount_log"]
    - previous_log_count
)


# Verify the features

new_hab_features = [
    "hab_cellcount_log",
    "hab_class_lag1",
    "hab_log_change_1wk"
]

print("HAB feature engineering completed!")
print("\nNew features created:")

for feature in new_hab_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Confirm no observations were removed
assert len(df_fe) == len(df)

# Preview results
display(
    df_fe[
        [
            "week_start",
            "hab_class",
            "hab_cellcount_max",
            "hab_cellcount_log",
            "hab_class_lag1",
            "hab_log_change_1wk"
        ]
    ].head(12)
)


HAB feature engineering completed!

New features created:
- hab_cellcount_log
- hab_class_lag1
- hab_log_change_1wk

Original columns: 47
Updated columns: 50

Original rows: 366
Updated rows: 366


,week_start,hab_class,hab_cellcount_max,hab_cellcount_log,hab_class_lag1,hab_log_change_1wk
0,2018-12-31,2.0,186266667.0,19.042690,NaN,NaN
1,2019-01-07,2.0,1303204.0,14.080337,2.0,-4.962353
2,2019-01-14,2.0,2428000.0,14.702579,2.0,0.622242
3,2019-01-21,2.0,282720.0,12.552216,2.0,-2.150363
4,2019-01-28,0.0,667.0,6.504288,2.0,-6.047928
5,2019-02-04,0.0,333.0,5.811141,0.0,-0.693147
6,2019-02-11,0.0,0.0,0.000000,0.0,-5.811141
7,2019-02-18,0.0,0.0,0.000000,0.0,0.000000
8,2019-02-25,0.0,500.0,6.216606,0.0,6.216606
9,2019-03-04,0.0,0.0,0.000000,0.0,-6.216606



## 2. Seasonal Feature Engineering

This section creates cyclical seasonal features to represent annual patterns that may influence Harmful Algal Bloom (HAB) severity.

Our EDA showed that HAB severity varies across different periods of the year. Seasonal information may help models capture recurring environmental conditions associated with bloom development.

To represent seasonality continuously, we create two features:

1. **season_sin:** Sine transformation of the day of year.
2. **season_cos:** Cosine transformation of the day of year.

### Benefits

- Capture recurring annual patterns in HAB severity.
- Represent the transition between December and January continuously.
- Provide seasonal information for linear and tree-based machine-learning models.

In [8]:
# Seasonal Feature Engineering

# Extract the day of year from the weekly date
day_of_year = df_fe["week_start"].dt.dayofyear

# Feature 4: Seasonal sine encoding
df_fe["season_sin"] = np.sin(
    2 * np.pi * (day_of_year - 1) / 365.25
)

# Feature 5: Seasonal cosine encoding
df_fe["season_cos"] = np.cos(
    2 * np.pi * (day_of_year - 1) / 365.25
)

new_seasonal_features = [
    "season_sin",
    "season_cos"
]

print("Seasonal feature engineering completed!")

print("\nNew features created:")
for feature in new_seasonal_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Verify no observations were removed
assert len(df_fe) == len(df)

# Verify both features stay between -1 and 1
assert df_fe[new_seasonal_features].abs().le(
    1.000001
).all().all()

# Preview seasonal features
display(
    df_fe[
        [
            "week_start",
            "hab_class",
            "season_sin",
            "season_cos"
        ]
    ].head(12)
)


Seasonal feature engineering completed!

New features created:
- season_sin
- season_cos

Original columns: 47
Updated columns: 52

Original rows: 366
Updated rows: 366


,week_start,hab_class,season_sin,season_cos
0,2018-12-31,2.0,-0.021501,0.999769
1,2019-01-07,2.0,0.103031,0.994678
2,2019-01-14,2.0,0.221772,0.975099
3,2019-01-21,2.0,0.337301,0.941397
4,2019-01-28,0.0,0.447945,0.894061
5,2019-02-04,0.0,0.552101,0.833777
6,2019-02-11,0.0,0.648262,0.761418
7,2019-02-18,0.0,0.735034,0.678031
8,2019-02-25,0.0,0.811160,0.584824
9,2019-03-04,0.0,0.875539,0.483147



## 3. Environmental Feature Engineering

This section creates environmental features that summarize recent conditions over multiple weeks.

### Features to Create

**1. air_tmean_4wk**

The average air temperature over the current week and previous three weeks.

This feature captures sustained temperature conditions and reduces the influence of short-term fluctuations.

**2. chl_anomaly_4wk**

The average coastal chlorophyll anomaly over the current week and previous three weeks.

This feature captures sustained chlorophyll conditions relative to the seasonal baseline.


In [9]:
# Environmental Feature Engineering

# Feature 6: Four-week average air temperature
df_fe["air_tmean_4wk"] = (
    df_fe.groupby("region")["air_tmean_c"]
    .transform(
        lambda x: x.rolling(
            window=4,
            min_periods=4
        ).mean()
    )
)

# Feature 7: Four-week average chlorophyll anomaly
df_fe["chl_anomaly_4wk"] = (
    df_fe.groupby("region")["chl_coastal_anomaly"]
    .transform(
        lambda x: x.rolling(
            window=4,
            min_periods=4
        ).mean()
    )
)

# Verify engineered features

new_environmental_features = [
    "air_tmean_4wk",
    "chl_anomaly_4wk"
]

print("Environmental feature engineering completed!")

print("\nNew features created:")
for feature in new_environmental_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Updated columns:", df_fe.shape[1])

print("\nOriginal rows:", df.shape[0])
print("Updated rows:", df_fe.shape[0])

# Verify no rows were removed
assert len(df_fe) == len(df)

# Preview original and engineered features
display(
    df_fe[
        [
            "week_start",
            "air_tmean_c",
            "air_tmean_4wk",
            "chl_coastal_anomaly",
            "chl_anomaly_4wk"
        ]
    ].head(12)
)


Environmental feature engineering completed!

New features created:
- air_tmean_4wk
- chl_anomaly_4wk

Original columns: 47
Updated columns: 54

Original rows: 366
Updated rows: 366


,week_start,air_tmean_c,air_tmean_4wk,chl_coastal_anomaly,chl_anomaly_4wk
0,2018-12-31,20.793651,NaN,0.061709,NaN
1,2019-01-07,17.142857,NaN,0.550108,NaN
2,2019-01-14,13.968254,NaN,0.264531,NaN
3,2019-01-21,14.761905,16.666667,0.539973,0.354080
4,2019-01-28,15.476190,15.337302,-0.594875,0.189934
5,2019-02-04,19.629630,15.958995,-0.181204,0.007106
6,2019-02-11,19.523810,17.347884,-0.313682,-0.137447
7,2019-02-18,23.452381,19.520503,-0.133152,-0.305728
8,2019-02-25,21.984127,21.147487,0.053866,-0.143543
9,2019-03-04,18.571429,20.882937,-0.045820,-0.109697



## 4. Highly Redundant Feature Analysis

This section identifies highly correlated numerical predictors that may provide overlapping information.

Removing unnecessary redundancy can:

- Simplify the feature set.
- Reduce multicollinearity in linear models.
- Improve model interpretability.
- Reduce unnecessary model complexity.
- Make feature selection more manageable for our small training dataset.

### Method

We calculate pairwise Pearson correlations between numerical predictor features using only the training dataset.

A correlation threshold of |r| >= 0.95 is used to flag highly redundant feature pairs for further review.

In [ ]:
# Highly Redundant Feature Analysis

# Select training observations only
train_fe = df_fe[
    df_fe["split"] == "train"
].copy()

print("Training rows:", len(train_fe))
print("Training period:",
      train_fe["week_start"].min(),
      "to",
      train_fe["week_start"].max())

# Exclude non-predictor columns

exclude_columns = [
    "y_class",
    "y_cellcount_max",
    "n_habsos_samples",
    "low_coverage",
    "has_nutrients",
    "has_ksrq",
    "has_mote",
    "cyano_samples"
]

# Select numerical predictor candidates
numeric_features = [
    col for col in train_fe.select_dtypes(
        include="number"
    ).columns
    if col not in exclude_columns
]

print("\nNumber of numerical predictors examined:",
      len(numeric_features))


# Find highly correlated pairs

correlation_threshold = 0.95
minimum_overlap = 30

high_corr_pairs = []

for i, feature_1 in enumerate(numeric_features):

    for feature_2 in numeric_features[i + 1:]:

        # Identify observations where both values exist
        valid = train_fe[
            [feature_1, feature_2]
        ].notna().all(axis=1)

        n_overlap = valid.sum()

        # Require enough paired observations
        if n_overlap < minimum_overlap:
            continue

        x = train_fe.loc[valid, feature_1]
        y = train_fe.loc[valid, feature_2]

        # Skip constant features
        if x.nunique() < 2 or y.nunique() < 2:
            continue

        correlation = x.corr(y)

        if abs(correlation) >= correlation_threshold:

            high_corr_pairs.append({
                "Feature 1": feature_1,
                "Feature 2": feature_2,
                "Correlation": round(correlation, 4),
                "Absolute Correlation": round(
                    abs(correlation), 4
                ),
                "Overlapping Observations": int(n_overlap)
            })

# Display results

high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=[
        "Feature 1",
        "Feature 2",
        "Correlation",
        "Absolute Correlation",
        "Overlapping Observations"
    ]
)

if not high_corr_df.empty:

    high_corr_df = high_corr_df.sort_values(
        "Absolute Correlation",
        ascending=False
    ).reset_index(drop=True)

    print("\nHIGHLY CORRELATED FEATURE PAIRS")
    print("-" * 50)

    display(high_corr_df)

else:
    print("\nNo feature pairs exceeded the threshold.")

print("\nCorrelation threshold:",
      correlation_threshold)

print("Minimum overlapping observations:",
      minimum_overlap)



Training rows: 157
Training period: 2018-12-31 00:00:00 to 2021-12-27 00:00:00

Number of numerical predictors examined: 43

HIGHLY CORRELATED FEATURE PAIRS
--------------------------------------------------


,Feature 1,Feature 2,Correlation,Absolute Correlation,Overlapping Observations
0,mote_temp_mean_c,mote_temp_max_c,0.9911,0.9911,73
1,mote_temp_mean_c,air_tmean_4wk,0.9708,0.9708,73
2,mote_temp_max_c,air_tmean_4wk,0.9667,0.9667,73
3,air_tmin_c,air_tmean_c,0.9641,0.9641,157
4,air_tmean_c,mote_temp_mean_c,0.9594,0.9594,73
5,air_tmax_c,air_tmean_c,0.9513,0.9513,157



Correlation threshold: 0.95
Minimum overlapping observations: 30



### Final Redundant Feature Removal

Remove selected redundant predictors from the feature-engineered dataset.

A Pearson correlation threshold of |r| >= 0.95 is used to flag highly correlated features for review.

#### Features Selected for Removal

| Feature Removed | Feature Retained | Justification |
|---|---|---|
| mote_temp_max_c | mote_temp_mean_c | Correlation = 0.9911; no unique training-week coverage |
| air_tmin_c | air_tmean_c | Correlation = 0.9641; no unique training-week coverage |
| air_tmax_c | air_tmean_c | Correlation = 0.9513; no unique training-week coverage |


In [ ]:
#  Final Redundant Feature Removal

# Remove only the three selected temperature features
features_to_remove = [
    "mote_temp_max_c",
    "air_tmin_c",
    "air_tmax_c"
]

# Create reduced dataset without modifying df_fe
df_fe_reduced = df_fe.drop(
    columns=features_to_remove
).copy()

# Verify both HAB representations are retained
assert "hab_cellcount_max" in df_fe_reduced.columns
assert "hab_cellcount_log" in df_fe_reduced.columns

# Verify both nitrogen measurements are retained
assert "nutr_tkn_mgL" in df_fe_reduced.columns
assert "nutr_tn_mgL" in df_fe_reduced.columns

# Verify dataset dimensions
assert df_fe_reduced.shape == (366, 51)

print("REDUNDANT FEATURE REMOVAL COMPLETED")
print("-" * 50)

print("\nFeatures removed:")
for feature in features_to_remove:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Full engineered columns:", df_fe.shape[1])
print("Reduced columns:", df_fe_reduced.shape[1])

print("\nOriginal rows:", len(df))
print("Reduced rows:", len(df_fe_reduced))


REDUNDANT FEATURE REMOVAL COMPLETED
--------------------------------------------------

Features removed:
- mote_temp_max_c
- air_tmin_c
- air_tmax_c

Original columns: 47
Full engineered columns: 54
Reduced columns: 51

Original rows: 366
Reduced rows: 366

Both HAB count representations retained.
Both nitrogen features retained.
All verification checks passed!



## Export Feature-Engineered Datasets

### Feature Engineering Summary

- Original dataset: 47 columns.
- New engineered features: 7.
- Complete engineered dataset: 54 columns.
- Redundant features removed: 3.
- Final reduced dataset: 51 columns.
- Observations preserved: 366.

### Redundancy Reduction

The following features were removed:

1. mote_temp_max_c
2. air_tmin_c
3. air_tmax_c


### Output Files

**model_table_weekly_fe_full.parquet**

Complete feature-engineered dataset containing 54 columns.

**model_table_weekly_fe_reduced.parquet**

Reduced feature-engineered dataset containing 51 columns.

### Data Integrity and Handoff

- All 366 weekly observations are preserved.
- Missing values remain unchanged.
- The original model_table_weekly.parquet file is not overwritten.

In [16]:
# Export Feature-Engineered Datasets

# Find the folder containing the original dataset
output_dir = Path(data_path).resolve().parent

# Define output file locations
full_output_path = (
    output_dir / "model_table_weekly_fe_full.parquet"
)

reduced_output_path = (
    output_dir / "model_table_weekly_fe_reduced.parquet"
)

# Make sure we are not overwriting the original file
original_path = Path(data_path).resolve()

assert full_output_path != original_path
assert reduced_output_path != original_path

# -----------------------------------
# Save the datasets
# -----------------------------------

df_fe.to_parquet(
    full_output_path,
    index=False
)

df_fe_reduced.to_parquet(
    reduced_output_path,
    index=False
)

# Reload files to verify the exports


full_check = pd.read_parquet(full_output_path)
reduced_check = pd.read_parquet(reduced_output_path)

# Verify dataset dimensions
assert full_check.shape == (366, 54)
assert reduced_check.shape == (366, 51)

# Verify saved data matches the notebook data
pd.testing.assert_frame_equal(
    full_check,
    df_fe
)

pd.testing.assert_frame_equal(
    reduced_check,
    df_fe_reduced
)

# Verify important retained features
important_features = [
    "nutr_tkn_mgL",
    "nutr_tn_mgL",
    "hab_cellcount_max",
    "hab_cellcount_log",
    "season_sin",
    "season_cos",
    "air_tmean_4wk",
    "chl_anomaly_4wk"
]

assert all(
    feature in reduced_check.columns
    for feature in important_features
)


# Display export results

print("DATASETS EXPORTED SUCCESSFULLY!")
print("-" * 50)

print("\nFull feature-engineered dataset:")
print(full_output_path)
print("Shape:", full_check.shape)

print("\nReduced feature-engineered dataset:")
print(reduced_output_path)
print("Shape:", reduced_check.shape)

print("\nAll verification checks passed!")
print("Original dataset was not overwritten.")
print("All 366 observations were preserved.")


DATASETS EXPORTED SUCCESSFULLY!
--------------------------------------------------

Full feature-engineered dataset:
/Users/mohammedchowdhury/Desktop/BTT/AI_BTT copy/Biointerphase-1B-copy/data/model_table_weekly_fe_full.parquet
Shape: (366, 54)

Reduced feature-engineered dataset:
/Users/mohammedchowdhury/Desktop/BTT/AI_BTT copy/Biointerphase-1B-copy/data/model_table_weekly_fe_reduced.parquet
Shape: (366, 51)

All verification checks passed!
Original dataset was not overwritten.
All 366 observations were preserved.
